# Milestone 5: Evaluation
Kevin — CSC 475, Music Maven (Group 4)

Objective 2, PI.3–PI.4:
- **PI.3**: Top-1 and top-5 accuracy on a held-out test set
- **PI.4**: Confusion matrix — which genres are most commonly misclassified as each other

**Evaluation methodology:** Only aggregated artist profiles are available (not individual songs),
so accuracy is measured at the genre level. An 80/20 train/test split is used; a prediction
is correct if the nearest neighbour(s) share the test artist's primary genre.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [ ]:
profiles = pd.read_csv('artist_profiles.csv')

# primary genre = first genre in the comma-separated list
profiles['primary_genre'] = (
    profiles['genres'].fillna('').str.split(',').str[0].str.strip()
)
labelled = profiles[profiles['primary_genre'] != ''].reset_index(drop=True)

n_genres = labelled['primary_genre'].nunique()
print(f'{len(profiles)} total artists, {len(labelled)} with genre labels')
print(f'{n_genres} unique primary genres')
labelled[['artist_id', 'artist_name', 'primary_genre', 'tempo', 'energy', 'valence']].head()

In [ ]:
MIR_FEATURES = ['tempo', 'energy', 'valence', 'danceability', 'key', 'mode']

def normalize_features(X, method='minmax'):
    X = np.array(X, dtype=float)
    if method == 'minmax':
        mins = X.min(axis=0); maxs = X.max(axis=0); ranges = maxs - mins
        with np.errstate(invalid='ignore', divide='ignore'):
            X_norm = np.where(ranges == 0, 0.0, (X - mins) / ranges)
        params = {'mins': mins, 'maxs': maxs}
    elif method == 'zscore':
        means = X.mean(axis=0); stds = X.std(axis=0)
        with np.errstate(invalid='ignore', divide='ignore'):
            X_norm = np.where(stds == 0, 0.0, (X - means) / stds)
        params = {'means': means, 'stds': stds}
    else:
        raise ValueError(f'unknown method: {method!r}')
    return X_norm, params

def apply_normalization(x, params, method='minmax'):
    x = np.array(x, dtype=float)
    if method == 'minmax':
        ranges = params['maxs'] - params['mins']
        with np.errstate(invalid='ignore', divide='ignore'):
            return np.where(ranges == 0, 0.0, (x - params['mins']) / ranges)
    elif method == 'zscore':
        with np.errstate(invalid='ignore', divide='ignore'):
            return np.where(params['stds'] == 0, 0.0, (x - params['means']) / params['stds'])
    else:
        raise ValueError(f'unknown method: {method!r}')

def prepare_weights(weights, n_features):
    if weights is None:
        return np.full(n_features, 1.0 / n_features)
    w = np.array(weights, dtype=float)
    total = w.sum()
    return np.full(n_features, 1.0 / n_features) if total == 0 else w / total

def weighted_euclidean_batch(query, candidates, weights=None):
    query = np.array(query, dtype=float)
    candidates = np.array(candidates, dtype=float)
    w = prepare_weights(weights, query.shape[0])
    diff = candidates - query
    return np.sqrt((w * diff * diff).sum(axis=1))

In [ ]:
class ArtistKNNClassifier:
    # k-NN classifier that maps a feature vector to the most likely artist(s)

    def __init__(self, profiles_df, k=5, features=None, weights=None, normalize='minmax'):
        self.k = k
        self.features = features if features is not None else list(MIR_FEATURES)
        self.normalize_method = normalize
        df = profiles_df.reset_index(drop=True)
        raw = df[self.features].to_numpy(dtype=float)
        self._artist_ids   = df['artist_id'].to_numpy()
        self._artist_names = df['artist_name'].to_numpy()
        self._weight_vec = None
        if weights is not None:
            self._weight_vec = np.array([weights.get(f, 0.0) for f in self.features], dtype=float)
        self._norm_matrix, self._norm_params = normalize_features(raw, method=normalize)

    def classify(self, song_features, k=None):
        k = k if k is not None else self.k
        if isinstance(song_features, np.ndarray):
            raw_vec = song_features.astype(float)
        elif isinstance(song_features, (dict, pd.Series)):
            raw_vec = np.array([float(song_features[f]) for f in self.features], dtype=float)
        else:
            raw_vec = np.array(song_features, dtype=float)
        q = apply_normalization(raw_vec, self._norm_params, method=self.normalize_method)
        dists = weighted_euclidean_batch(q, self._norm_matrix, self._weight_vec)
        return self._majority_vote(dists, k)

    def _majority_vote(self, distances, k):
        fi = np.where(np.isfinite(distances))[0]
        if len(fi) == 0:
            return []
        eff_k = min(k, len(fi))
        nn_idx = fi[np.argsort(distances[fi])[:eff_k]]
        artist_data = {}
        for aid, aname, dist in zip(self._artist_ids[nn_idx],
                                    self._artist_names[nn_idx],
                                    distances[nn_idx]):
            if aid not in artist_data:
                artist_data[aid] = {'artist_id': aid, 'artist_name': aname,
                                    'votes': 0, 'total_dist': 0.0}
            artist_data[aid]['votes'] += 1
            artist_data[aid]['total_dist'] += dist
        results = []
        for e in artist_data.values():
            v = e['votes']
            results.append({'artist_id': e['artist_id'], 'artist_name': e['artist_name'],
                            'votes': v, 'probability': v / eff_k,
                            'avg_distance': e['total_dist'] / v})
        results.sort(key=lambda x: (-x['votes'], x['avg_distance']))
        return results

## PI.3: Top-1 and Top-5 Accuracy
Evaluate the k-NN classifier on a held-out 20% test set. Each test artist's profile acts as a
proxy query against the training set. A result is correct if the predicted artist's primary
genre matches the test artist's primary genre.

In [ ]:
train_df, test_df = train_test_split(labelled, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f'Train: {len(train_df)} artists | Test: {len(test_df)} artists')

In [ ]:
# build classifier on training data; k=10 gives enough neighbours to evaluate top-5
clf = ArtistKNNClassifier(train_df, k=10)
train_genre_map = train_df.set_index('artist_id')['primary_genre'].to_dict()

top1_hits = top5_hits = 0
true_labels, pred1_labels = [], []

for _, row in test_df.iterrows():
    preds = clf.classify(row)
    if not preds:
        continue
    true_g  = row['primary_genre']
    pred1_g = train_genre_map.get(preds[0]['artist_id'], '')
    top1_hits += (pred1_g == true_g)
    top5_genres = [train_genre_map.get(p['artist_id'], '') for p in preds[:5]]
    top5_hits += (true_g in top5_genres)
    true_labels.append(true_g)
    pred1_labels.append(pred1_g)

n = len(true_labels)
top1_acc = top1_hits / n
top5_acc = top5_hits / n
print(f'Evaluated {n} test artists')
print(f'Top-1 accuracy (genre match): {top1_acc:.3f}  ({top1_hits}/{n})')
print(f'Top-5 accuracy (genre match): {top5_acc:.3f}  ({top5_hits}/{n})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# left: overall top-1 / top-5
ax = axes[0]
bars = ax.bar(['Top-1', 'Top-5'], [top1_acc, top5_acc],
              color=['steelblue', 'darkorange'], width=0.4, edgecolor='black')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy (genre match)')
ax.set_title('k-NN Genre-Level Accuracy\n(80/20 split, k=10)')
for bar, v in zip(bars, [top1_acc, top5_acc]):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.03,
            f'{v:.3f}', ha='center', va='bottom', fontsize=12)

# right: per-genre top-1 accuracy for genres with >= 10 test samples
genre_hits  = defaultdict(int)
genre_total = defaultdict(int)
for t, p in zip(true_labels, pred1_labels):
    genre_total[t] += 1
    genre_hits[t]  += (p == t)

genre_acc = {g: genre_hits[g] / genre_total[g]
             for g in genre_total if genre_total[g] >= 10}
top20 = sorted(genre_acc.items(), key=lambda x: -x[1])[:20]
g_labels, g_accs = zip(*top20)

ax2 = axes[1]
ax2.barh(g_labels[::-1], g_accs[::-1], edgecolor='black', alpha=0.75, color='steelblue')
ax2.axvline(top1_acc, color='red', linestyle='--', linewidth=1.5,
            label=f'Overall ({top1_acc:.3f})')
ax2.set_xlim(0, 1.15)
ax2.set_xlabel('Top-1 Accuracy')
ax2.set_title('Per-Genre Accuracy (>= 10 test artists, top 20)')
ax2.legend(fontsize=8)

fig.tight_layout()
plt.show()

## PI.4: Confusion Matrix
For the 12 most frequent genres in the test set, show how often each is predicted correctly
(diagonal) versus confused with another genre (off-diagonal). A second chart lists the most
common specific genre-to-genre error pairs.

In [ ]:
top_genres = [g for g, _ in Counter(true_labels).most_common(12)]

filt = [(t, p) for t, p in zip(true_labels, pred1_labels)
        if t in top_genres and p in top_genres]
true_f, pred_f = zip(*filt) if filt else ([], [])

cm = confusion_matrix(true_f, pred_f, labels=top_genres)

fig, ax = plt.subplots(figsize=(13, 10))
im = ax.imshow(cm, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(top_genres)))
ax.set_yticks(range(len(top_genres)))
ax.set_xticklabels(top_genres, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(top_genres, fontsize=9)
ax.set_xlabel('Predicted Genre (top-1)', fontsize=11)
ax.set_ylabel('True Genre', fontsize=11)
ax.set_title('PI.4 — Confusion Matrix: Top 12 Genres', fontsize=13)
thresh = cm.max() * 0.5
for i in range(len(top_genres)):
    for j in range(len(top_genres)):
        if cm[i, j] > 0:
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8,
                    color='white' if cm[i, j] > thresh else 'black')
plt.tight_layout()
plt.show()

In [ ]:
errors = [(t, p) for t, p in zip(true_labels, pred1_labels) if t != p]
error_counts = Counter(errors).most_common(15)

if error_counts:
    labels_e = [f'{t} -> {p}' for (t, p), _ in error_counts]
    counts_e  = [c for _, c in error_counts]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(labels_e[::-1], counts_e[::-1], edgecolor='black', alpha=0.75, color='salmon')
    ax.set_xlabel('Misclassification Count')
    ax.set_title('Top 15 Genre Misclassification Pairs (True -> Predicted)')
    plt.tight_layout()
    plt.show()